# АНАЛИЗ — EDA перед разработкой кредитного скоринга

Ноутбук ожидает CSV `NEW_SAMPLE_with_OOT.csv` в той же папке. Он формирует полный EDA до обучения: качество, пропуски, временную стабильность, зависимость переменных с TARGET, ассоциации и рекомендации.

## Легенда: как читать этот ноутбук

Это не обучение модели, а последовательная проверка, **какие данные у нас есть, можно ли им доверять и какие признаки безопасно нести в скоринг**.

| Этап | Что происходит | Зачем это нужно |
|---|---|---|
| 1. Загрузка и качество | Считаем строки, пропуски, уникальность и дубликаты | Находим технические ошибки и потенциальные утечки до моделирования. |
| 2. Словарь полей | Объясняем смысл каждой переменной и ограничения её использования | Чтобы не спутать ID/PII с полезными предикторами. |
| 3. Анализ каждой переменной | Смотрим распределение, частоту и event rate | Понимаем, есть ли в признаке сигнал для риска. |
| 4. Взаимодействия | Сравниваем признаки друг с другом и с TARGET | Ищем дублирование, смешение сегментов и риск утечки. |
| 5. Решение перед обучением | Фиксируем, что подготовить для baseline и OOT | Делаем следующий шаг воспроизводимым. |

**Как читать результаты:** `event rate` — доля строк с `TARGET = 1` среди размеченных строк. Сравнивайте его между группами только вместе с числом заявок: редкая категория может давать случайно высокий/низкий показатель. Строки с пустым `TARGET` не участвуют в event rate и проверяются отдельно.

## Словарь переменных

Ниже приведена интерпретация по названиям полей и содержимому файла. Перед финальным обучением её нужно подтвердить у владельца данных, особенно смысл `LIM` и бизнес-определение `TARGET`.

| Переменная | Значение | Роль в анализе / модели |
|---|---|---|
| `PARTNER` | Партнёр или организация-источник заявки | Категориальный кандидат; проверять временной дрейф. |
| `APPID` | Уникальный идентификатор заявки | Технический ID, исключить из модели. |
| `APPDATE` | Дата и время подачи заявки | Используется для временного анализа и OOT-разделения. |
| `DOCSERNUM` | Идентификатор/номер документа клиента | PII: не использовать в сыром виде; допустима только частотность. |
| `MOBILEPHONE` | Идентификатор мобильного телефона клиента | PII: не использовать в сыром виде; допустима только частотность. |
| `EMAIL` | Идентификатор e-mail клиента | PII: не использовать в сыром виде; допустима только частотность. |
| `LIM` | Сумма кредитного лимита или запрашиваемой суммы кредита* | Числовой признак; проверяются нули, выбросы и нелинейность. |
| `PRODUCTTYPE` | Тип кредитного продукта | Категориальный кандидат. |
| `CHANNEL` | Канал оформления/продажи | Категориальный кандидат. |
| `MODEL` | Модель или описание товара/предмета кредитования | Высококардинальный признак; нужны обработка редких уровней и пропусков. |
| `TARGET` | Результат целевого события: `1` — событие, `0` — несобытие, пусто — нет разметки | Целевая переменная; строки без разметки отдельно контролируются. |

\* Точное бизнес-значение `LIM` следует подтвердить до построения модели.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")

csv_files = sorted(Path.cwd().glob("*.csv"))
assert csv_files, "Положите CSV в одну папку с ноутбуком."
DATA_PATH = next((p for p in csv_files if "NEW_SAMPLE" in p.name.upper()), csv_files[0])

df = pd.read_csv(DATA_PATH, sep=";", encoding="cp1251", low_memory=False)
df["APPDATE_DT"] = pd.to_datetime(df["APPDATE"], format="%d.%m.%Y %H:%M:%S", errors="coerce")
df["APP_MONTH"] = df["APPDATE_DT"].dt.to_period("M").astype("string")
labeled = df[df["TARGET"].isin([0, 1])].copy()

print(f"Файл: {DATA_PATH.name}")
print(f"Наблюдений: {len(df):,}; размечено: {len(labeled):,} ({len(labeled)/len(df):.1%})")
display(df.head())

## 1. Загрузка и контроль качества

Сначала проверяем, что файл прочитан корректно и не содержит проблем, способных исказить модель. `APPID` рассматривается как технический ключ: его уникальность полезна для контроля качества, но сам ID не должен стать признаком.

## 1. Качество данных и роли переменных

In [ ]:
base_cols = [c for c in df.columns if c not in ["APPDATE_DT", "APP_MONTH"]]
quality = pd.DataFrame({
    "dtype": df[base_cols].dtypes.astype(str),
    "missing_n": df[base_cols].isna().sum(),
    "missing_%": (df[base_cols].isna().mean() * 100).round(2),
    "unique_n": df[base_cols].nunique(dropna=False),
    "unique_%": (df[base_cols].nunique(dropna=False) / len(df) * 100).round(2),
}).sort_values(["missing_%", "unique_n"], ascending=[False, False])
display(quality)

print("Полные дубликаты:", f"{df.duplicated().sum():,}")
print("Дубли APPID:", f"{df.APPID.duplicated().sum():,}")
print("Некорректные APPDATE:", f"{df.APPDATE_DT.isna().sum():,}")

plt.figure(figsize=(10, 4))
quality["missing_%"].sort_values(ascending=False).plot.bar(color="#D55E00")
plt.title("Доля пропусков по переменным"); plt.ylabel("%"); plt.xticks(rotation=45, ha="right"); plt.show()

roles = pd.DataFrame([
    ("APPID", "ID заявки", "исключить из модели"),
    ("APPDATE", "время", "календарные признаки; OOT-разделение"),
    ("DOCSERNUM / MOBILEPHONE / EMAIL", "PII/идентификаторы", "не использовать сырыми; допустима частота"),
    ("PARTNER / PRODUCTTYPE / CHANNEL", "низкая кардинальность", "кандидаты в категориальные признаки"),
    ("LIM", "числовой", "нули, выбросы, log1p"),
    ("MODEL", "высокая кардинальность", "редкие уровни/агрегации; не сырой код"),
    ("TARGET", "цель", "не включать в признаки"),
], columns=["variable", "роль", "рекомендация"])
display(roles)

## 2. Поочерёдный анализ переменных

Каждый блок отвечает на один вопрос: **как устроена переменная, насколько она полна и меняется ли event rate вместе с ней?**

### `TARGET` и `APPDATE`

TARGET задаёт событие для скоринга, а дата заявки показывает, насколько результат стабилен во времени. Это основа корректного OOT-разделения.

## 2. TARGET, время и OOT

In [ ]:
display(pd.DataFrame({
    "count": df.TARGET.value_counts(dropna=False),
    "share_%": (df.TARGET.value_counts(dropna=False, normalize=True) * 100).round(2),
}))
print(f"Event rate на размеченной части: {labeled.TARGET.mean():.2%}")
print("Период заявок:", df.APPDATE_DT.min(), "—", df.APPDATE_DT.max())

monthly = df.groupby("APP_MONTH", dropna=False).agg(
    applications=("APPID", "size"),
    labeled=("TARGET", lambda s: s.isin([0, 1]).sum()),
    event_rate=("TARGET", "mean"),
    target_missing=("TARGET", lambda s: s.isna().mean()),
).reset_index()
display(monthly)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
sns.barplot(data=monthly, x="APP_MONTH", y="applications", color="#4C78A8", ax=axes[0])
axes[0].set(title="Количество заявок по месяцам", xlabel="", ylabel="заявок")
sns.lineplot(data=monthly, x="APP_MONTH", y="event_rate", marker="o", label="event rate", ax=axes[1])
sns.lineplot(data=monthly, x="APP_MONTH", y="target_missing", marker="o", label="TARGET отсутствует", ax=axes[1])
axes[1].set(title="Стабильность таргета во времени", xlabel="месяц", ylabel="доля")
for ax in axes: ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

### `LIM`, `PARTNER`, `PRODUCTTYPE`, `CHANNEL`, `MODEL` и PII-поля

Сначала проверяем форму распределения `LIM`, затем по отдельности сравниваем объём и event rate категорий. PII не раскрываются: для них допустим только анализ частотности, если это согласовано с политикой данных.

## 3. Распределения и зависимости с TARGET

In [ ]:
# LIM
print("Доля нулевого LIM:", f"{df.LIM.eq(0).mean():.2%}")
display(df.LIM.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame("LIM"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df.LIM.clip(upper=df.LIM.quantile(.99)), bins=60, ax=axes[0])
axes[0].set(title="LIM, обрезан на 99-м перцентиле")
sns.histplot(np.log1p(df.LIM), bins=60, ax=axes[1]); axes[1].set(title="log(1 + LIM)")
plt.tight_layout(); plt.show()

labeled["LIM_BIN"] = pd.qcut(labeled.LIM, q=10, duplicates="drop")
lim_target = labeled.groupby("LIM_BIN", observed=True).agg(
    applications=("TARGET", "size"), event_rate=("TARGET", "mean"), median_LIM=("LIM", "median")
).reset_index()
display(lim_target)
plt.figure(figsize=(10, 4))
sns.barplot(data=lim_target, x="LIM_BIN", y="event_rate", color="#E45756")
plt.title("Event rate по децилям LIM"); plt.xticks(rotation=45, ha="right"); plt.show()

def category_target(data, col, top_n=30):
    w = data[[col, "TARGET"]].copy()
    w[col] = w[col].fillna("<MISSING>").astype(str)
    z = w.groupby(col).agg(applications=("TARGET", "size"), event_rate=("TARGET", "mean"))
    z["share_%"] = z.applications / len(w) * 100
    return z.sort_values("applications", ascending=False).head(top_n).reset_index()

LOW_CARD = ["PARTNER", "PRODUCTTYPE", "CHANNEL"]
for col in LOW_CARD:
    z = category_target(labeled, col)
    print(f"### {col}"); display(z)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, .35 * len(z))))
    sns.barplot(data=z, y=col, x="applications", color="#4C78A8", ax=axes[0])
    sns.barplot(data=z.sort_values("event_rate"), y=col, x="event_rate", color="#E45756", ax=axes[1])
    axes[0].set(title=f"{col}: объём"); axes[1].set(title=f"{col}: event rate")
    plt.tight_layout(); plt.show()

model_top = category_target(labeled, "MODEL", top_n=25)
display(model_top)
plt.figure(figsize=(12, 8))
sns.barplot(data=model_top.sort_values("event_rate"), y="MODEL", x="event_rate", color="#E45756")
plt.title("MODEL: event rate у 25 наиболее частых уровней"); plt.show()

# PII не раскрываются: исследуется только частотность значений.
for col in ["DOCSERNUM", "MOBILEPHONE", "EMAIL", "MODEL"]:
    freq = df[col].fillna("<MISSING>").value_counts()
    w = labeled[[col, "TARGET"]].copy()
    w["frequency"] = w[col].fillna("<MISSING>").map(freq)
    w["bucket"] = pd.cut(w["frequency"], [0, 1, 2, 5, 10, np.inf], labels=["1", "2", "3–5", "6–10", "11+"])
    z = w.groupby("bucket", observed=True).agg(applications=("TARGET", "size"), event_rate=("TARGET", "mean"))
    print(f"Частотность {col}, без раскрытия значений"); display(z)

## 3. Взаимодействия переменных

Отдельный хороший признак может дублировать другой или работать лишь в отдельном сегменте. Pearson/Spearman показывают связь `LIM` с TARGET, а Cramér V — силу связи категориальных полей между собой и с TARGET. Это ориентир для проверки, не автоматическое правило отбора.

## 4. Корреляции, ассоциации и действия перед обучением

In [ ]:
pearson = labeled[["LIM", "TARGET"]].corr(method="pearson")
spearman = labeled[["LIM", "TARGET"]].corr(method="spearman")
print("Pearson"); display(pearson)
print("Spearman"); display(spearman)
sns.heatmap(pearson, annot=True, vmin=-1, vmax=1, cmap="vlag", square=True)
plt.title("Pearson-корреляция LIM и TARGET"); plt.show()

def cramers_v(x, y):
    t = pd.crosstab(x.fillna("<MISSING>"), y)
    if min(t.shape) < 2: return np.nan
    chi2 = chi2_contingency(t, correction=False)[0]
    n = t.to_numpy().sum()
    phi2 = chi2 / n
    r, k = t.shape
    phi2 = max(0, phi2 - (k - 1) * (r - 1) / max(n - 1, 1))
    r = r - (r - 1) ** 2 / max(n - 1, 1)
    k = k - (k - 1) ** 2 / max(n - 1, 1)
    return np.sqrt(phi2 / max(min(r - 1, k - 1), 1e-12))

association = pd.DataFrame([
    {"feature": col, "Cramér V with TARGET": cramers_v(labeled[col], labeled.TARGET)}
    for col in LOW_CARD
]).sort_values("Cramér V with TARGET", ascending=False)
display(association)

matrix = pd.DataFrame(index=LOW_CARD, columns=LOW_CARD, dtype=float)
for a in LOW_CARD:
    for b in LOW_CARD:
        matrix.loc[a, b] = 1 if a == b else cramers_v(df[a], df[b])
sns.heatmap(matrix, annot=True, vmin=0, vmax=1, cmap="Blues", square=True)
plt.title("Cramér V между категориальными признаками"); plt.show()

print("""
Перед обучением:
1. Подтвердить definition of default, окно наблюдения и смысл пустого TARGET.
2. Исключить APPID и сырые PII; frequency/target encoding делать только внутри train-fold.
3. Делить train/validation/OOT строго по APPDATE и проверять временной дрейф.
4. Обработать MODEL (missing/редкие уровни) и LIM (нули/выбросы).
5. Оценивать ROC-AUC/Gini, PR-AUC и калибровку; начать с логистической регрессии и сравнить с бустингом на общем OOT.
""")

## 4. Что делать после EDA

Следующий шаг — зафиксировать definition of default и окно наблюдения, исключить утечки, подготовить признаки только на train-fold и сравнить интерпретируемый baseline с бустингом на одном OOT-наборе.